# Policy target evaluation visualization

This notebook visualizes `policy_target_summary.csv` for the four policy methods, three target tasks, and five seeds. It reports raw data sanity checks, method/target means with 95% confidence intervals, per-seed behavior, metric heatmaps, and a compact ranking table.

The file path is resolved relative to either the repository root or this notebook's directory.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

# Resolve the CSV when launched from the repo root, output directory, or another child directory.
cwd = Path.cwd().resolve()
candidates = [cwd / 'policy_target_summary.csv']
for parent in (cwd, *cwd.parents):
    candidates.append(parent / 'outputs' / 'results' / 'dr_policy_target_eval' / 'policy_target_summary.csv')
data_path = next((path for path in dict.fromkeys(candidates) if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError('Could not find outputs/results/dr_policy_target_eval/policy_target_summary.csv')

df = pd.read_csv(data_path)
print(f'Loaded {len(df):,} rows from {data_path}')
print(f'seaborn available: {HAS_SEABORN}')

In [ ]:
required_columns = {
    'method', 'seed', 'target', 'mean_reward', 'success_rate',
    'mean_steps', 'goal_dist_mean', 'wm_shaped_reward'
}
missing = sorted(required_columns.difference(df.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

numeric_columns = [column for column in df.columns if column not in {'method', 'target', 'wm_checkpoint', 'policy_checkpoint', 'evaluation_csv'}]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='raise')
print('Shape:', df.shape)
print('Methods:', sorted(df['method'].unique()))
print('Targets:', sorted(df['target'].unique()))
print('Seeds:', sorted(df['seed'].unique()))
print('Missing values in key metrics:')
display(df[sorted(required_columns)].isna().sum().to_frame('missing'))
print('Rows per method/target (expected: five seeds each):')
display(df.groupby(['method', 'target'])['seed'].nunique().unstack(fill_value=0))

In [ ]:
# Aggregate over seeds. The interval is a normal-approximation 95% CI of the seed mean.
metric_specs = {
    'mean_reward': ('Mean reward', 'higher'),
    'success_rate': ('Success rate', 'higher'),
    'mean_steps': ('Mean steps', 'lower'),
    'goal_dist_mean': ('Mean goal distance', 'lower'),
    'wm_shaped_reward': ('WM-shaped reward', 'higher'),
}

def aggregate(group):
    result = {'n_seeds': group['seed'].nunique()}
    for metric in metric_specs:
        values = group[metric].dropna().to_numpy(dtype=float)
        result[f'{metric}_mean'] = values.mean() if len(values) else np.nan
        sem = values.std(ddof=1) / math.sqrt(len(values)) if len(values) > 1 else 0.0
        result[f'{metric}_ci95'] = 1.96 * sem
    return pd.Series(result)

summary_rows = []
for (method, target), group in df.groupby(['method', 'target'], sort=True):
    row = aggregate(group).to_dict()
    row.update({'method': method, 'target': target})
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)[['method', 'target'] + [column for column in summary_rows[0] if column not in {'method', 'target'}]]
display(summary.round(4))

In [ ]:
# Aggregate comparison: one panel per key metric, with a 95% CI across seeds.
plot_metrics = ['mean_reward', 'success_rate', 'mean_steps', 'goal_dist_mean']
methods = list(df['method'].drop_duplicates())
targets = list(df['target'].drop_duplicates())
colors = plt.get_cmap('tab10')(np.linspace(0, 1, max(len(targets), 1)))
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, metric in zip(axes.flat, plot_metrics):
    x = np.arange(len(methods))
    width = 0.8 / max(len(targets), 1)
    for target_index, target in enumerate(targets):
        part = summary[summary['target'] == target].set_index('method').reindex(methods)
        offset = (target_index - (len(targets) - 1) / 2) * width
        ax.bar(
            x + offset, part[f'{metric}_mean'], width,
            yerr=part[f'{metric}_ci95'], capsize=3,
            label=target, color=colors[target_index], alpha=0.85,
        )
    ax.set_title(metric_specs[metric][0])
    ax.set_xticks(x, methods)
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
axes[0, 0].legend(title='Target', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.suptitle('Policy target evaluation: mean ± 95% CI across seeds', fontsize=15)
plt.show()

In [ ]:
# Per-seed curves make variability and outlier seeds visible.
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, metric in zip(axes.flat, plot_metrics):
    for target_index, target in enumerate(targets):
        for method in methods:
            part = df[(df['target'] == target) & (df['method'] == method)].sort_values('seed')
            style = '-' if target_index == 0 else ('--' if target_index == 1 else ':')
            ax.plot(part['seed'], part[metric], marker='o', linestyle=style,
                    color=plt.get_cmap('tab10')(methods.index(method)),
                    alpha=0.8, label=f'{method} / {target}')
    ax.set_title(metric_specs[metric][0])
    ax.set_xlabel('Seed')
    ax.grid(alpha=0.25)
axes[0, 0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.suptitle('Per-seed policy results', fontsize=15)
plt.show()

In [ ]:
# Heatmaps show the method × target structure at a glance.
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, metric in zip(axes.flat, plot_metrics):
    table = summary.pivot(index='method', columns='target', values=f'{metric}_mean').reindex(index=methods, columns=targets)
    if HAS_SEABORN:
        sns.heatmap(table, annot=True, fmt='.3f', cmap='viridis', ax=ax, cbar=False)
    else:
        image = ax.imshow(table.to_numpy(), aspect='auto', cmap='viridis')
        ax.set_xticks(range(len(table.columns)), table.columns)
        ax.set_yticks(range(len(table.index)), table.index)
        for row in range(table.shape[0]):
            for column in range(table.shape[1]):
                ax.text(column, row, f'{table.iloc[row, column]:.3f}', ha='center', va='center', color='white')
    ax.set_title(metric_specs[metric][0])
fig.suptitle('Mean metrics by policy method and target', fontsize=15)
plt.show()

In [ ]:
# Ranking is shown per target. Primary ordering is success rate, then reward,
# with fewer steps as a tie-breaker. This keeps target difficulty visible.
ranking = summary[['method', 'target', 'success_rate_mean', 'mean_reward_mean', 'mean_steps_mean']].copy()
ranking['success_rank'] = ranking.groupby('target')['success_rate_mean'].rank(method='min', ascending=False)
ranking['reward_rank'] = ranking.groupby('target')['mean_reward_mean'].rank(method='min', ascending=False)
ranking['overall_rank'] = (ranking['success_rank'] + ranking['reward_rank']) / 2
ranking = ranking.sort_values(['target', 'overall_rank', 'mean_steps_mean', 'method'])
display(ranking.rename(columns={
    'success_rate_mean': 'success_rate',
    'mean_reward_mean': 'mean_reward',
    'mean_steps_mean': 'mean_steps',
}).round(4))